In [ ]:
#%pip install openpyxl
#%oio install pandas


In [2]:
import pandas as pd
import numpy as np
import glob
import re
from pathlib import Path


In [3]:
## 2000-2010: https://agreste.agriculture.gouv.fr/agreste-web/disaron/SAANR_DEVELOPPE_2/detail/
## 2010: 

In [2]:
# -----------------------------
# SETTINGS
# -----------------------------
CROPYIELD_DIR = Path("/Users/marielouiselysholt/Desktop/Master/Masters_2026/EDA/Cropyield")

EARLY_CROP = "Blé tendre d'hiver"  # see note below
LATE_CROP  = "01 - Blé tendre d'hiver et épeautre"

LATE_FILE  = CROPYIELD_DIR / "SAA_2010-2024_définitives_donnees_departementales.xlsx"
LATE_SHEET = "COP"

OUTPUT_FILE = "ble_tendre_hiver_yield_2000_2024_regions_mainland.csv"

# Mainland region codes (metropolitan France + Corsica)
MAINLAND_REGION_CODES = {
    "11",  # Île-de-France
    "24",  # Centre-Val de Loire
    "27",  # Bourgogne-Franche-Comté
    "28",  # Normandie
    "32",  # Hauts-de-France
    "44",  # Grand Est
    "52",  # Pays de la Loire
    "53",  # Bretagne
    "75",  # Nouvelle-Aquitaine
    "76",  # Occitanie
    "84",  # Auvergne-Rhône-Alpes
    "93",  # Provence-Alpes-Côte d’Azur
    "94",  # Corse (remove if you don't want Corsica)
}

# -----------------------------
# Helper functions
# -----------------------------

def clean_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace("\u00a0", "", regex=False).str.strip()
    s = s.str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(s, errors="coerce")


def norm_region_code_early(series: pd.Series) -> pd.Series:
    """
    Early CSV: REGION column looks like 'NR01', 'NR11', ... or '...'
    -> return '01', '11', ... or NaN
    """
    s = series.astype(str).str.strip()
    s = s.replace("...", np.nan)
    # extract 2-digit code after 'NR'
    code = s.str.extract(r"NR(\d{2})", expand=False)
    return code


def extract_region_code_from_LIB_REG2(series: pd.Series) -> pd.Series:
    """
    Late Excel: LIB_REG2 like '11 - Île-de-France'
    -> return '11'
    """
    s = series.astype(str).str.strip()
    code = s.str.extract(r"^(\d{2})", expand=False)
    return code


def find_header_row(excel_path: Path, sheet: str, needle: str = "LIB_SAA", max_scan: int = 80) -> int:
    raw = pd.read_excel(excel_path, sheet_name=sheet, header=None, dtype=str)
    for i in range(min(max_scan, len(raw))):
        if raw.iloc[i].astype(str).str.contains(needle, na=False).any():
            return i
    raise ValueError(f"Could not find header row containing '{needle}' in sheet '{sheet}'.")

# -----------------------------
# EARLY FILES (2000–2010) - REGION LEVEL
# -----------------------------

early_files = sorted(glob.glob(str(CROPYIELD_DIR / "FDS_DEVELOPPE_*.csv")))
early_list = []

for f in early_files:
    year_match = re.search(r"(20\d{2})", Path(f).name)
    if not year_match:
        continue
    year = int(year_match.group(1))
    if year > 2010:
        continue

    df = pd.read_csv(f, sep=";", encoding="latin1", dtype=str)

    # Filter crop
    # NOTE: depending on encoding, the exact label may look garbled.
    # You may need to inspect df["N306_LIB"].unique() and adjust EARLY_CROP if needed.
    df["N306_LIB"] = df["N306_LIB"].astype(str).str.strip()
    df = df[df["N306_LIB"] == EARLY_CROP].copy()
    if df.empty:
        continue

    # Drop aggregates like REGION == '...'
    df["REGION"] = df["REGION"].astype(str).str.strip()
    df = df[df["REGION"] != "..."].copy()

    # Keep only production + surface
    df["N027_LIB"] = df["N027_LIB"].astype(str).str.strip()
    df = df[df["N027_LIB"].isin(["Production (volume)", "Superficie développée"])].copy()
    if df.empty:
        continue

    # Numeric
    df["VALEUR"] = clean_numeric(df["VALEUR"])

    # classify
    df["_type"] = np.where(df["N027_LIB"].eq("Production (volume)"), "prod", "surf")

    # region code (2-digit), year
    df["reg"] = norm_region_code_early(df["REGION"])
    df["year"] = pd.to_numeric(df["ANNREF"], errors="coerce")

    # keep only mainland regions
    df = df.dropna(subset=["reg", "year"])
    df = df[df["reg"].isin(MAINLAND_REGION_CODES)].copy()

    # aggregate duplicates then pivot (region-year)
    g = (
        df.groupby(["year", "reg", "_type"], as_index=False)["VALEUR"]
          .sum()
    )

    wide = (
        g.pivot_table(index=["year", "reg"], columns="_type", values="VALEUR", aggfunc="sum")
         .reset_index()
    )
    wide.columns.name = None

    wide["prod"] = wide.get("prod")
    wide["surf"] = wide.get("surf")
    wide["yield"] = np.where((wide["surf"] > 0) & wide["prod"].notna(), wide["prod"] / wide["surf"], np.nan)
    wide["source"] = "early_csv"

    early_list.append(wide[["year", "reg", "prod", "surf", "yield", "source"]])

early_panel = pd.concat(early_list, ignore_index=True) if early_list else pd.DataFrame(
    columns=["year", "reg", "prod", "surf", "yield", "source"]
)

# -----------------------------
# LATE FILE (2010–2024) - REGION LEVEL
# -----------------------------

header_row = find_header_row(LATE_FILE, LATE_SHEET, needle="LIB_SAA")
late = pd.read_excel(LATE_FILE, sheet_name=LATE_SHEET, header=header_row, dtype=str)

# Crop filter
late = late[late["LIB_SAA"].astype(str).str.strip() == LATE_CROP].copy()

# Region code
late["reg"] = extract_region_code_from_LIB_REG2(late["LIB_REG2"])

# Keep only mainland regions
late = late[late["reg"].isin(MAINLAND_REGION_CODES)].copy()

# Years where both SURF_YYYY and PROD_YYYY exist
surf_years = {int(c.split("_")[1]) for c in late.columns if re.fullmatch(r"SURF_\d{4}", str(c))}
prod_years = {int(c.split("_")[1]) for c in late.columns if re.fullmatch(r"PROD_\d{4}", str(c))}
years = sorted(surf_years & prod_years)

late_rows = []
for y in years:
    surf_col = f"SURF_{y}"
    prod_col = f"PROD_{y}"

    temp = late[["reg", surf_col, prod_col]].copy()
    temp["year"] = y
    temp["surf"] = clean_numeric(temp[surf_col])
    temp["prod"] = clean_numeric(temp[prod_col])
    temp["yield"] = np.where((temp["surf"] > 0) & temp["prod"].notna(), temp["prod"] / temp["surf"], np.nan)
    temp["source"] = "late_excel"

    late_rows.append(temp[["year", "reg", "prod", "surf", "yield", "source"]])

late_panel = pd.concat(late_rows, ignore_index=True) if late_rows else pd.DataFrame(
    columns=["year", "reg", "prod", "surf", "yield", "source"]
)

# -----------------------------
# MERGE PANELS (regions, mainland only)
# Prefer late from 2011 onward
# -----------------------------

panel = pd.concat([
    early_panel[early_panel["year"] < 2011],
    late_panel[late_panel["year"] >= 2011]
])

panel = panel.sort_values(["year", "reg"]).reset_index(drop=True)

panel.to_csv(OUTPUT_FILE, index=False)

print("Panel created:", OUTPUT_FILE)
if not panel.empty:
    print("Years:", panel["year"].min(), "-", panel["year"].max())
    print("Regions (codes):", sorted(panel["reg"].unique()))

Panel created: ble_tendre_hiver_yield_2000_2024_regions_mainland.csv
Years: 2011 - 2024
Regions (codes): ['11', '24', '27', '28', '32', '44', '52', '53', '75', '76', '84', '93', '94']


/var/folders/g6/v5w89nm96b732r3v5w91v8dm0000gn/T/ipykernel_17186/2424215449.py:193: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  panel = pd.concat([
